# OPTIMA — Kaggle: Heavy Hugging Face LLM Enrichment Experiment

Compares Hugging Face causal LMs as the **enrichment** stage of the Optima pipeline:

```
GitHub base.json (frozen snapshot)
        |
        v
  LLM enrichment (gated: load -> trivial -> 1 function -> 3 functions -> full 96)
        |
        v
  embeddings (optima.rag, unchanged) -> FAISS indexes
        |
        v
  retrieval (unchanged) -> Recall@K / MRR evaluation
```

**How to use this notebook**

1. Edit the `CONFIG` cell: `BASE_JSON_URL` and `MODEL_NAME` (one Hugging Face model per run).
2. *Run All*. Each model must pass GATES 1-4 before the full 96-function run starts.
3. For a run that may take hours, use **Save & Run All (Commit)** so the checkpoint
   survives a lost session; resume with `RESUME_INPUT_DIR` pointed at the previous
   version's output dataset.
4. The analyzer / libclang are **not** used here: this notebook consumes an existing
   `base.json` and never regenerates it.
5. Every enrichment model is evaluated against the exact same base.json snapshot and
   the exact same frozen benchmark, so `summaries/comparison.csv` is a fair comparison.
6. **Multi-GPU:** on a session with two or more GPUs (e.g. Kaggle's T4 x2), a model
   that does not fit on one GPU is automatically sharded across all of them via
   accelerate (`device_map="auto"` + an explicit `max_memory`), not left unused --
   `check_fit()` decides this at runtime and GATE 1 prints the resulting
   `model.hf_device_map` so you can confirm both GPUs are in use for a 30B-class
   model. A model that fits on one GPU stays on one GPU even in a multi-GPU
   session. On a single-GPU session everything degrades to plain single-GPU loading.


## Cell 1 — Configuration

In [19]:
import os
from pathlib import Path

# ---- Optima repository (this project) ----
OPTIMA_REPO_URL = "https://github.com/I1gorr/optima_python.git"
OPTIMA_REF = "main"  # pin to a commit SHA for full reproducibility

# ---- Filesystem roots ----
OPTIMA_DIR = Path("/kaggle/working/optima-python")
OUTPUT_ROOT = Path("/kaggle/working/optima_outputs")

# ---- base.json source: point this at your own analyzer output on GitHub.
# Prefer a commit-SHA raw URL (not a branch) for reproducibility, or pin
# BASE_JSON_EXPECTED_SHA256 below.
BASE_JSON_URL = "https://raw.githubusercontent.com/I1gorr/optima_python/main/output/base.json"
BASE_JSON_EXPECTED_SHA256 = None       # set to a sha256 hex digest to pin an exact snapshot
EXPECTED_FUNCTION_COUNT = 96           # set to None if your base.json legitimately differs

# ---- Benchmark (generated once from base.json only; never regenerated per-model) ----
BENCHMARK_JSON_URL = None              # None = generate from base.json; or a curated queries.json URL
NUM_QUERIES = 20
BENCHMARK_SEED = 42

# ---- Model: the ONLY thing you change to test a different Hugging Face model.
# One notebook run enriches with exactly one model -- there is no registry,
# queue, or model-selection logic. check_fit()/load_model_safe() always shard
# this one model instance across every visible GPU (e.g. both T4s) rather than
# placing it on a single GPU, even when it would fit alone on one.
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
QUANTIZATION = "nf4"                      # "nf4" (needs bitsandbytes) or "fp16"

# ---- Generation ----
PROMPT_VARIANT = "colab_v2_no_module_ir"  # excludes AST/CFG/module-level LLVM IR by default,
                                           # keeping each function's prompt in the ~500-1500 token range
MAX_INPUT_TOKENS = 1500                   # per-function prompt budget; the reduction ladder in
                                           # build_bounded_messages() trims further if still over this
MAX_NEW_TOKENS = None                     # None = no artificial output cap; the model generates
                                           # until it emits EOS or exhausts its own context window
                                           # (see models.max_new_tokens_for_context). This is a pure
                                           # model-comparison run: set an int here only if you want to
                                           # deliberately compare models under a fixed output budget.
DO_SAMPLE = False                         # greedy decoding for deterministic, reproducible output
TEMPERATURE = None
TOP_P = None
RETRIES = 2

# ---- Full-run safety ----
# There is no consecutive-failure/success-rate circuit breaker: the run never
# stops because of a model's OUTPUT (invalid_json, insufficient context, a
# schema-incomplete response, ...) -- every one of the 96 functions is recorded
# and the run continues. Only a genuine infrastructure failure (an unrecovered
# CUDA OOM, a model/tokenizer that fails to load, ...) stops it, by raising.
MATERIALIZE_EVERY = 1                     # write enriched_<slug>.json after EVERY function (not batched);
                                           # raise this only if per-function I/O (re-reading/re-writing
                                           # the whole base.json-sized artifact each time) becomes a
                                           # bottleneck on a very large dataset -- reliability over speed
SMOKE_FUNCTION_IDS = None                 # e.g. ["path.cpp::func::12"] to pin the smoke-test functions
RUN_FULL_ENRICHMENT = True

# ---- Embedding / retrieval / evaluation (runs only after the GPU is free of any LLM) ----
RUN_EVALUATION = True
EMBEDDING_MODELS = ["bge-small"]
REPRESENTATION_MODES = ["hybrid", "semantic"]
K = 10

# ---- Resume (attach a previous version's /kaggle/working/optima_outputs as a dataset) ----
RESUME_INPUT_DIR = None   # e.g. Path("/kaggle/input/optima-outputs-v1/optima_outputs")

# ---- Environment / cleanup ----
INSTALL_BITSANDBYTES_IF_MISSING = True
DELETE_MODEL_CACHE_AFTER_UNLOAD = True

HANDLE = None  # the only variable ever allowed to hold a loaded model

# The exact slug ModelSpec will use for this MODEL_NAME (matches
# optima.rag.embedding_simple._slug, which models.spec_from_model_id() calls
# later); computed here, before any optima_kaggle import, only so the run
# directory's manifest can record which model this run is for.
import re as _re
MODEL_SLUG = _re.sub(r"[^a-zA-Z0-9._-]+", "-", MODEL_NAME.strip()).strip("-").lower()

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# CUDA_VISIBLE_DEVICES is deliberately left unset: both GPUs (e.g. a Kaggle
# T4 x2 session) must stay visible so a model that needs sharding can use
# them both. torch.cuda.device_count(), read in the diagnostics cell below,
# is what the rest of this notebook branches on -- never an env var.
os.environ.setdefault("HF_HOME", "/tmp/hf_home")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OPTIMA_EMBEDDING_DEVICE", "cuda")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "warning")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Configuration loaded. MODEL_NAME={MODEL_NAME!r} (slug={MODEL_SLUG!r})")


Configuration loaded.


## Cell 2 — Kaggle environment diagnostics (pre-clone)

In [20]:
import platform
import subprocess

print(f"Python version: {platform.python_version()}")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout)
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")


Python version: 3.12.13
Tesla T4, 15360 MiB, 14807 MiB, 580.159.04
Tesla T4, 15360 MiB, 14807 MiB, 580.159.04



## Cell 3 — Clone / checkout Optima Python (sparse, shallow; no pip install)

In [21]:
import subprocess
import sys


def _run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


if not (OPTIMA_DIR / ".git").is_dir():
    if OPTIMA_DIR.exists():
        raise RuntimeError(f"{OPTIMA_DIR} exists but is not a git repository; remove it first.")
    _run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
          OPTIMA_REPO_URL, str(OPTIMA_DIR)])
    _run(["git", "sparse-checkout", "set", "--cone", "optima", "colab", "optima_kaggle"], cwd=OPTIMA_DIR)
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)
else:
    status = _run(["git", "status", "--porcelain"], cwd=OPTIMA_DIR)
    if status.stdout.strip():
        raise RuntimeError(f"{OPTIMA_DIR} has uncommitted changes; refusing to touch it.")
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)

# A fresh kernel never has stale modules, but a re-run of this cell might.
for name in list(sys.modules):
    if name == "optima" or name.startswith(("optima.", "optima_kaggle", "colab")):
        del sys.modules[name]
if str(OPTIMA_DIR) not in sys.path:
    sys.path.insert(0, str(OPTIMA_DIR))

OPTIMA_COMMIT = _run(["git", "rev-parse", "HEAD"], cwd=OPTIMA_DIR).stdout.strip()
print(f"Optima checked out at {OPTIMA_DIR}, commit {OPTIMA_COMMIT}")


$ git status --porcelain
$ git fetch --depth 1 origin main
$ git checkout FETCH_HEAD
$ git rev-parse HEAD
Optima checked out at /kaggle/working/optima-python, commit 4df19fba405b963767eef97e36fd346dccc8636e


## Cell 4 — Install/check Python dependencies (never touches torch or clang)

In [22]:
from optima_kaggle import environment as env

DEPS = env.ensure_python_deps(allow_install=True)


  langchain_core           1.3.1
  langchain_community      0.4.2
  langchain_text_splitters unknown
  langchain_huggingface    unknown
  faiss                    1.15.1
  numpy                    2.0.2
  accelerate               1.13.0
  sentence_transformers    5.4.1
  tqdm                     4.67.3
  transformers             5.0.0
  matplotlib               3.10.0
  packaging                26.1
  huggingface_hub          1.11.0


## Cell 5 — GPU/CUDA/internet diagnostics

In [23]:
ENV_INFO = env.diagnose(hf_home=os.environ.get("HF_HOME"), working_dir="/kaggle/working")
NUM_GPUS = ENV_INFO["num_gpus"]
print(f"NUM_GPUS = {NUM_GPUS} (multi_gpu_capable={ENV_INFO['multi_gpu_capable']})")
env.gpu_report("session start")


  python_version             3.12.13
  torch_version              2.10.0+cu128
  cuda_available             True
  cuda_version               12.8
  cpu_ram_gib                31.35
  internet_reachable         True
  num_gpus                   2
  multi_gpu_capable          True
  gpu[0]                   Tesla T4, cc=7.5, free=14.46/14.562 GiB
  gpu[1]                   Tesla T4, cc=7.5, free=14.46/14.562 GiB
  gpu_name                   Tesla T4
  compute_capability         7.5
  gpu_total_vram_gib         14.562
  gpu_free_vram_gib          14.46
  working_dir_disk_free_gib  19.43
  hf_home_disk_free_gib      1024.32
NUM_GPUS = 2 (multi_gpu_capable=True)
[gpu:session start] gpu0: alloc=0.0 free=14.4598/14.5622 | gpu1: alloc=0.0 free=14.4598/14.5622 || total_alloc=0.0, min_free=14.4598


{'label': 'session start',
 'gpus': [{'index': 0,
   'allocated_gib': 0.0,
   'reserved_gib': 0.0,
   'max_allocated_gib': 0.0,
   'free_gib': 14.4598,
   'total_gib': 14.5622},
  {'index': 1,
   'allocated_gib': 0.0,
   'reserved_gib': 0.0,
   'max_allocated_gib': 0.0,
   'free_gib': 14.4598,
   'total_gib': 14.5622}],
 'aggregate': {'allocated_gib': 0.0,
  'reserved_gib': 0.0,
  'max_allocated_gib': 0.0,
  'free_gib': 14.4598,
  'total_gib': 29.1244},
 'timestamp': 1789635059.6383593}

## Cell 6 — bitsandbytes compatibility probe (fails loud, never silently downgrades)

In [24]:
BNB = env.probe_bitsandbytes(INSTALL_BITSANDBYTES_IF_MISSING)
print(f"bitsandbytes: ok={BNB.ok}, stage={BNB.stage}, version={BNB.version}")
print(BNB.message)


bitsandbytes: ok=True, stage=ok, version=0.50.2
bitsandbytes nf4 is usable on this GPU.


## Cell 7 — Import Optima + optima_kaggle; verify the analyzer/libclang are not loaded

In [25]:
import inspect
import sys as _sys

from colab import colab_pipeline
from optima.rag import embedding_simple
from optima_kaggle import enrichment, models, retrieval_eval, snapshot

assert "optima.analyzer" not in _sys.modules, "the analyzer must not be imported for this experiment"
assert "clang" not in _sys.modules and "clang.cindex" not in _sys.modules, "libclang must not be imported"

assert ".tmp" in inspect.getsource(colab_pipeline.save_json), (
    "colab_pipeline.save_json is not atomic on this checkout; push the local fix to "
    "colab/colab_pipeline.py on GitHub before running this notebook."
)
assert "OPTIMA_EMBEDDING_DEVICE" in inspect.getsource(embedding_simple), (
    "optima.rag.embedding_simple lacks OPTIMA_EMBEDDING_DEVICE support on this checkout; "
    "push the local fix on GitHub before running this notebook (otherwise embeddings run on CPU)."
)
print("Imports OK; analyzer/libclang not loaded; required upstream fixes are present.")


Imports OK; analyzer/libclang not loaded; required upstream fixes are present.


## Cell 8 — Download and snapshot base.json (downloaded exactly once per run)

In [26]:
SNAP = snapshot.download_base_snapshot(BASE_JSON_URL, OUTPUT_ROOT, expected_sha256=BASE_JSON_EXPECTED_SHA256)

CTX = snapshot.RunContext.create(
    OUTPUT_ROOT, SNAP,
    config={
        "prompt_variant": PROMPT_VARIANT, "max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE, "top_p": TOP_P, "retries": RETRIES,
        "embedding_models": EMBEDDING_MODELS, "representation_modes": REPRESENTATION_MODES, "k": K,
    },
    env=ENV_INFO, optima_commit=OPTIMA_COMMIT, configured_models=[MODEL_SLUG],
)
print(f"Run directory: {CTX.run_dir}")


BASE SNAPSHOT: base-3200fa96e3c5 | 80,166,344 bytes | https://raw.githubusercontent.com/I1gorr/optima_python/main/output/base.json
Run directory: /kaggle/working/optima_outputs/runs/base-3200fa96e3c5


## Cell 9 — Validate base.json

In [27]:
BASE_REPORT = snapshot.validate_base_snapshot(SNAP.path, EXPECTED_FUNCTION_COUNT)


Number of nodes: 96
Available fields: analysis_status, ast, basic_blocks, called_by, calls, cfg, compiler_error, dependencies, id, llvm, llvm_ir, mangled_name, name, parameters, qualified_name, return_type, source, source_code, source_location
Nested structure: project -> files[] -> functions[]
Enrichment fields already present: none
Input file: /kaggle/working/optima_outputs/runs/base-3200fa96e3c5/base/base.json
Malformed files: 0
Malformed nodes: 0
Malformed enrichment objects: 0
Validation: OK
Status: BASE JSON
-> enrichment will run
base.json validated: 43 files, 96 functions, analysis_status={'llvm_function_not_found': 47, 'success': 40, 'source_only': 9}


## Cell 10 — Restore resumable state from a previous run (no-op if RESUME_INPUT_DIR is None)

In [28]:
snapshot.restore_resume_state(CTX, RESUME_INPUT_DIR)


## Cell 11 — Freeze the benchmark (generated once from base.json only; never from enriched JSON)

In [29]:
BENCH = snapshot.freeze_benchmark(CTX, NUM_QUERIES, BENCHMARK_SEED, BENCHMARK_JSON_URL)


INFO:optima.rag.evaluation:Loaded 20 benchmark queries from /kaggle/working/optima_outputs/runs/base-3200fa96e3c5/benchmark/queries.json


BENCHMARK: reusing frozen benchmark (20 queries) from /kaggle/working/optima_outputs/runs/base-3200fa96e3c5/benchmark/queries.json


---
## Model stages (`MODEL_NAME`)

GATE 1 (load) -> GATE 2 (trivial generation) -> GATE 3 (one real function) ->
GATE 4 (three real functions) -> full 96-function run. A failed gate raises and
stops here; it does not fall through to the full run.

## Cell 12 — Resolve the model spec, generation settings, and pre-load VRAM fit estimate

In [30]:
# One model, built directly from MODEL_NAME -- no registry, no queue.
# check_fit() (below) is the only feasibility gate: it decides single-GPU vs
# sharded-across-both-T4s placement for THIS model at runtime and prints a
# per-GPU fit table; there is no registry tier to consult either way.
SPEC = models.spec_from_model_id(
    MODEL_NAME, quantization=QUANTIZATION,
    max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
)
assert SPEC.slug == MODEL_SLUG  # sanity: matches the slug computed in the config cell
print(f"Model: {SPEC.model_id}  (slug={SPEC.slug}, quantization={SPEC.quantization}, "
      f"max_input_tokens={SPEC.max_input_tokens}, max_new_tokens={SPEC.max_new_tokens})")

GEN = enrichment.GenerationSettings(
    do_sample=DO_SAMPLE, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    retries=RETRIES, prompt_variant=PROMPT_VARIANT,
)
FIT = models.check_fit(SPEC, num_gpus=NUM_GPUS)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-32B-Instruct/5ede1c97bbab6ce5cda5812749b4c0bdf79b18dd/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-32B-Instruct/5ede1c97bbab6ce5cda5812749b4c0bdf79b18dd/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

FIT ESTIMATE for qwen25-32b-instruct-nf4 (2 GPU(s)): weights=18.305 GiB, total_headroom=2.487 GiB
  gpu[0]: free=14.46 GiB, reserved=1.244 GiB, budget=12.493 GiB
  gpu[1]: free=14.46 GiB, reserved=1.244 GiB, budget=12.493 GiB
  decision: placement=sharded, fits=True


## Cell 13 — GATE 1: model loads onto the GPU

In [35]:
try:
    env.gpu_report("before load", CTX.gpu_log_path)
    HANDLE = models.load_model_safe(SPEC, BNB)
    env.gpu_report("after load", CTX.gpu_log_path)
    # load_model_safe() already printed the full hf_device_map; this is a
    # one-line summary so a sharded 30B-class load is obvious at a glance.
    print(f"placement={HANDLE.load_report['gpu_placement']}, "
          f"gpus_used={HANDLE.load_report['gpu_count_used']}, "
          f"used_cpu_offload={HANDLE.load_report['used_cpu_offload']}")
    G1 = enrichment.gate1_load(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-32B-Instruct/5ede1c97bbab6ce5cda5812749b4c0bdf79b18dd/config.json "HTTP/1.1 200 OK"


[gpu:before load] gpu0: alloc=6.6122 free=7.4442/14.5622 | gpu1: alloc=11.3075 free=2.8152/14.5622 || total_alloc=17.9197, min_free=2.8152
FIT ESTIMATE for qwen25-32b-instruct-nf4 (2 GPU(s)): weights=18.305 GiB, total_headroom=2.487 GiB
  gpu[0]: free=7.444 GiB, reserved=1.244 GiB, budget=5.828 GiB
  gpu[1]: free=2.815 GiB, reserved=1.244 GiB, budget=1.431 GiB
  decision: placement=none, fits=False
MODEL UNLOADED: Qwen/Qwen2.5-32B-Instruct, delete_cache=False


ModelDoesNotFitError: qwen25-32b-instruct-nf4 does not fit: weights=18.305 GiB, total_headroom=2.487 GiB, per-GPU free=[7.44, 2.82] GiB across 2 GPU(s). CPU offload was not enabled for this model (allow_cpu_offload=False). Try: lower max_input_tokens, confirm nf4 quantization is selected, attach a second GPU, or (if appropriate for this model) enable allow_cpu_offload.

## Cell 14 — GATE 2: trivial generation succeeds

In [37]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 2 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    G2 = enrichment.gate2_trivial(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


AttributeError: 'NoneType' object has no attribute 'spec'

## Cell 15 — GATE 3: one real function from base.json is enriched

In [38]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 3 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    G3 = enrichment.gate3_one_function(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


AttributeError: 'NoneType' object has no attribute 'spec'

## Cell 16 — GATE 4: three real functions are enriched (including the largest bounded prompt)

In [55]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 4 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    # Load model -> enrich 3 functions -> verify enrichment fields ->
    # verify the checkpoint was written. Same MAX_NEW_TOKENS as the full run.
    G4 = enrichment.gate4_three_functions(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


GenerationOOMError: CUDA OOM generating for function src/search/see.cpp::see::14 (input_tokens=3355, max_new_tokens=768). Aborting this model's run rather than continuing with corrupted state.

## Cell 17 — Full resumable 96-function enrichment (only runs if GATES 1-4 passed)

In [ ]:
FULL = None
try:
    if RUN_FULL_ENRICHMENT:
        if HANDLE is None:
            raise RuntimeError(
                "Full enrichment requires a successfully loaded model HANDLE. "
                "GATE 1 must complete successfully first."
            )
        FULL = enrichment.run_full_enrichment(
            CTX, HANDLE, GEN, SNAP, materialize_every=MATERIALIZE_EVERY,
        )
        _metrics = FULL["metrics"]
        print()
        print("ENRICHMENT COMPLETE")
        print()
        print(f"Model: {MODEL_NAME}")
        print(f"Total functions: {_metrics.get('functions_total')}")
        print(f"Enriched: {_metrics.get('functions_enriched')}")
        print(f"Skipped/resumed (already done before this run): {FULL.get('resumed_functions', 0)}")
        print(f"Failed: {_metrics.get('functions_failed')}")
        print()
        print(f"Output:\n{FULL['enriched_json']}")
        print()
        print(_metrics)
    else:
        print("RUN_FULL_ENRICHMENT is False; skipping the full run.")
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 18 — Unload the model and verify GPU memory is released

In [ ]:
models.unload_model(HANDLE, delete_cache=DELETE_MODEL_CACHE_AFTER_UNLOAD)
HANDLE = None
env.gpu_report("after unload", CTX.gpu_log_path)


---
## Embedding, retrieval, evaluation

Everything below runs with **no LLM on the GPU** and reuses the existing
`optima.rag` embedding/retrieval/evaluation implementation unchanged.

## Cell 20 — Discover which corpora passed enrichment; verify the GPU is free of any LLM

In [ ]:
assert HANDLE is None, "A model handle is still live; unload it before embedding/retrieval."
env.assert_gpu_clean(threshold_gib=0.3)
CORPORA = retrieval_eval.discover_passed_corpora(CTX)
print("Corpora to embed/evaluate:", CORPORA)


## Cell 21 — Build FAISS indexes for raw + every passed enrichment model

In [ ]:
INDEXES = None
if RUN_EVALUATION:
    INDEXES = retrieval_eval.build_indexes(CTX, CORPORA, EMBEDDING_MODELS, REPRESENTATION_MODES)


## Cell 22 — Retrieval smoke test (pipeline sanity, not a quality bar)

In [ ]:
if RUN_EVALUATION:
    from optima.rag.embedding_simple import embedding_alias
    retrieval_eval.retrieval_smoke(
        CTX, CORPORA, embedding_alias(EMBEDDING_MODELS[0]), REPRESENTATION_MODES[0], BENCH
    )


## Cell 23 — Full retrieval evaluation (Recall@K, MRR, latency) for every mode/alias/corpus

In [ ]:
MATRICES = None
if RUN_EVALUATION:
    MATRICES = retrieval_eval.evaluate_all(CTX, EMBEDDING_MODELS, REPRESENTATION_MODES, BENCH, K, CORPORA)


## Cell 24 — Cross-model comparison table (same base.json + benchmark for every row)

In [ ]:
COMPARISON = None
if RUN_EVALUATION:
    COMPARISON = retrieval_eval.build_comparison(CTX, MATRICES, CORPORA, BENCH)
    try:
        import pandas as pd
        display(pd.DataFrame(COMPARISON["rows"]).sort_values(
            ["representation_mode", "embedding_alias", "mrr"], ascending=[True, True, False]
        ))
    except ImportError:
        for row in COMPARISON["rows"]:
            print(row)

    def _fmt(value):
        return f"{value:.4f}" if isinstance(value, (int, float)) else str(value)

    print()
    print("EVALUATION SUMMARY")
    print(f"Model: {MODEL_NAME}")
    for row in COMPARISON["rows"]:
        if row["corpus"] == MODEL_SLUG:
            print(f"  [{row['representation_mode']}/{row['embedding_alias']}] "
                  f"Recall@5={_fmt(row['recall_at_5'])}  MRR={_fmt(row['mrr'])}  "
                  f"Recall@10={_fmt(row['recall_at_10'])}")
    print(f"Full results: {CTX.summaries_dir / 'comparison.csv'}")


## Cell 25 — Final artifact report (raises if any configured model did not pass)

In [ ]:
snapshot.final_report(CTX)
